# Welcome to your first assignment!

Instructions are below. Please give this a try, and look in the solutions folder if you get stuck (or feel free to ask me!)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get to the assignment --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code below from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [3]:
# imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

In [4]:
# Constants

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL = "llama3.2:1b" # "llama3.2"

In [6]:
# Create a messages list using the same format that we used for OpenAI

from urllib.request import urlopen, Request
import xml.etree.ElementTree as ET

system_prompt = (
    """You are an impartial analyst. Assess the current national security threat to Estonia from Russia (conventional or hybrid) or Russia-influenced unrest. Use only the information provided by the user. Be concise, factual, and evidence-driven.

    Output (JSON only)
    {
      "assessment_time_utc": "<ISO8601>",
      "overall_threat_level": "LOW|MODERATE|SUBSTANTIAL|SEVERE|CRITICAL",
      "confidence": 0.0,
      "dimension_scores": {
        "military_posture": 0.0,
        "cross_border_coercion": 0.0,
        "cyber": 0.0,
        "disinformation": 0.0,
        "domestic_unrest": 0.0,
        "critical_infra": 0.0
      },
      "key_signals": ["", ""],
      "notes": ""
    }
    Threat Level Definitions

    LOW – Hostile action highly unlikely. Only background noise.
    MODERATE – Possible but not likely. Mild cyber/disinfo or routine border probes.
    SUBSTANTIAL – Likely. Sustained cyber, disinfo, or coercive actions; some GPS jamming.
    SEVERE – Highly likely. Coordinated multi-vector actions or sabotage indicators.
    CRITICAL – Imminent or ongoing attack. Rapid force movement, destructive cyber, verified plots.
    
    Scoring (0–5)
    
    0 = no threat signal, 5 = extreme signal.
    Overall level ≈ weighted average:
    LOW (<1), MODERATE (1–1.7), SUBSTANTIAL (1.8–2.6), SEVERE (2.6–3.3), CRITICAL (≥3.4).
    If any dimension ≥4.5 and corroborated → escalate to CRITICAL."""
)

class RssFeed:
    def __init__(self, url, timeout=10):
        self.url = url
        self.items = []
        self.text = ""

        req = Request(url, headers={"User-Agent": "StdlibRSS/1.0"})
        with urlopen(req, timeout=timeout) as r:
            data = r.read()

        root = ET.fromstring(data)
        channel = root.find(".//channel") or root  # fallback if no <channel>

        parts = []
        for it in channel.findall(".//item"):
            title = (it.findtext("title") or "").strip()
            author = (it.findtext("author") or "").strip()
            pub_date = (it.findtext("pubDate") or "").strip()
            description = (it.findtext("description") or "").strip()

            entry_str = (
                f"Title: {title}\n"
                f"Author: {author}\n"
                f"Date: {pub_date}\n"
                f"Description: {description}"
            )

            parts.append(entry_str)
            self.items.append({
                "title": title,
                "author": author,
                "pubDate": pub_date,
                "description": description
            })

        # join into single blob of text
        self.text = "\n\n".join(parts)

def make_user_prompt(feed: "RssFeed") -> str:
    """Format user prompt from RssFeed object."""
    return (
        f"""You are receiving the latest intelligence and situational indicators for Estonia.
        Score the threat level based on the provided facts. Facts:
        {feed.text}
        Please return a JSON assessment."""
        )

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": make_user_prompt(feed = RssFeed("https://news.postimees.ee/rss"))}
]

In [7]:
payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False
    }

In [8]:
# Let's just make sure the model is loaded

!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠏ pulling manifest ⠙ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 2.2 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 4.9 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 6.4 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 9.1 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  11 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  13 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  16 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  19 MB/2.0 GB                  pulling manifes

In [9]:
# If this doesn't work for any reason, try the 2 versions in the following cells
# And double check the instructions in the 'Recap on installation of Ollama' at the top of this lab
# And if none of that works - contact me!

response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)
print(response.json()['message']['content'])

{
  "assessment_time_utc": "<ISO8601>",
  "overall_threat_level": "SUBSTANTIAL",
  "confidence": 0.5,
  "dimension_scores": {
    "military_posture": 0.2,
    "cross_border_coercion": 0.3,
    "cyber": 0.4,
    "disinformation": 0.6,
    "domestic_unrest": 0.7,
    "critical_infra": 0.8
  },
  "key_signals": ["", "", "", "", ""],
  "notes": ""
}

Threat Level Assessment:

The current national security threat to Estonia from Russia is substantial, with multiple indicators suggesting a high likelihood of conventional or hybrid attacks.

* Three Russian fighter jets entered Estonian airspace without authorization, which is a clear violation of the country's sovereignty and territorial integrity.
* The Russian Orthodox clergy has publicly supported Russia's war against Ukraine, which indicates a shift in Moscow's diplomatic and military support for its proxy state.
* The defense forces have a conflict of interest at sea, with Estonia's naval vessels encountering Russian unmanned surface ve

# Introducing the ollama package

And now we'll do the same thing, but using the elegant ollama python package instead of a direct HTTP call.

Under the hood, it's making the same call as above to the ollama server running at localhost:11434

In [10]:
import ollama

response = ollama.chat(model=MODEL, messages=messages)
print(response['message']['content'])

{
  "assessment_time_utc": "<ISO8601>",
  "overall_threat_level": "SEVERE",
  "confidence": 0.8,
  "dimension_scores": {
    "military_posture": 2.4,
    "cross_border_coercion": 2.7,
    "cyber": 3.1,
    "disinformation": 2.9,
    "domestic_unrest": 2.5,
    "critical_infra": 0
  },
  "key_signals": ["", ""],
  "notes": ""
}

The overall threat level to Estonia is SEVERE, indicating that there is a significant and imminent threat of attack or sabotage. The key signals include:

* Three Russian fighter jets entering Estonian airspace without authorization (Three Russian fighter jets entered Estonian airspace without authorization)
* The Moscow Patriarchate supporting the Kremlin's war against Ukraine (Russian Orthodox clergypersons with Estonian residence permits support the Kremlin’s war against Ukraine)
* The expulsion of an Estonian diplomat by Russia (Russia expels Estonian diplomat)

The dimension scores indicate that the threat is substantial, with cyber and disinformation being

## Alternative approach - using OpenAI python library to connect to Ollama

In [13]:
# There's actually an alternative approach that some people might prefer
# You can use the OpenAI client python library to call Ollama:

from openai import OpenAI
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

response = ollama_via_openai.chat.completions.create(
    model=MODEL,
    messages=messages
)

print(response.choices[0].message.content)

{
  "assessment_time_utc": "<ISO8601>",
  "overall_threat_level": "SEVERE|CRITICAL",
  "confidence": 0.99,
  "dimension_scores": {
    "military_posture": 5,
    "cross_border_coercion": 3,
    "cyber": 4,
    "disinformation": 2,
    "domestic_unrest": 1,
    "critical_infra": 5
  },
  "key_signals": [" THREE Russian fighter jets entered Estonian airspace without authorization","],
  "notes": ""
}

Threat level assessment: The current threat level to Estonia is SEVERE|CRITICAL due to the violation of its airspace by three Russian fighter jets, which is a clear indicator of an attack. The use of GPS jamming and possible diversion of the drones from their intended path have been reported, increasing the likelihood of a hostile action.

Reasoning:

* The presence of multiple Russian fighter jets in Estonian airspace without authorization is a clear indication of an attack, and indicates a high level of commitment to defending Estonian airspace.
* The GPS jamming used by the drones has no

## Are you confused about why that works?

It seems strange, right? We just used OpenAI code to call Ollama?? What's going on?!

Here's the scoop:

The python class `OpenAI` is simply code written by OpenAI engineers that makes calls over the internet to an endpoint.  

When you call `openai.chat.completions.create()`, this python code just makes a web request to the following url: "https://api.openai.com/v1/chat/completions"

Code like this is known as a "client library" - it's just wrapper code that runs on your machine to make web requests. The actual power of GPT is running on OpenAI's cloud behind this API, not on your computer!

OpenAI was so popular, that lots of other AI providers provided identical web endpoints, so you could use the same approach.

So Ollama has an endpoint running on your local box at http://localhost:11434/v1/chat/completions  
And in week 2 we'll discover that lots of other providers do this too, including Gemini and DeepSeek.

And then the team at OpenAI had a great idea: they can extend their client library so you can specify a different 'base url', and use their library to call any compatible API.

That's it!

So when you say: `ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')`  
Then this will make the same endpoint calls, but to Ollama instead of OpenAI.

## Also trying the amazing reasoning model DeepSeek

Here we use the version of DeepSeek-reasoner that's been distilled to 1.5B.  
This is actually a 1.5B variant of Qwen that has been fine-tuned using synethic data generated by Deepseek R1.

Other sizes of DeepSeek are [here](https://ollama.com/library/deepseek-r1) all the way up to the full 671B parameter version, which would use up 404GB of your drive and is far too large for most!

In [11]:
!ollama pull deepseek-r1:1.5b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 1.7 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 3.7 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏ 6.7 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏ 9.2 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏  10 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏  13 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏  16 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   2% ▕                  ▏  17 MB/1.1 GB                  pulling manifes

In [14]:
# This may take a few minutes to run! You should then see a fascinating "thinking" trace inside <think> tags, followed by some decent definitions

response = ollama_via_openai.chat.completions.create(
    model="deepseek-r1:1.5b",
    messages=[{"role": "user", "content": "Please give definitions of some core concepts behind LLMs: a neural network, attention and the transformer"}]
)

print(response.choices[0].message.content)

<think>
Okay, so I'm trying to understand what LLMs are. From what I've read before, they're something with big models trained on lots of data probably. But when it comes to the specifics like neural networks, attention, and transformers, my knowledge is a bit hazy. Let me break this down step by step.

First off, I know that an LLM is short for Large Language Model. It's designed to understand and generate human-level language. So they're used in various applications like chatbots, summarizing articles, predicting stock trends... it must be really versatile.

Now, let's get into core concepts: neural networks, attention, transformers.

Starting with neural networks because I think that's the broad term here. From what I remember, neural networks are inspired by the human brain. They have layers of nodes connected in a way that processes information through these layers, adjusting connections or weights based on data patterns. This allows them to learn and model complex data.

So for a

# NOW the exercise for you

Take the code from day1 and incorporate it here, to build a website summarizer that uses Llama 3.2 running locally instead of OpenAI; use either of the above approaches.

In [29]:
from urllib.request import urlopen, Request
import xml.etree.ElementTree as ET

class RssFeed:
    def __init__(self, url, timeout=10):
        self.url = url
        self.items = []
        self.text = ""

        req = Request(url, headers={"User-Agent": "StdlibRSS/1.0"})
        with urlopen(req, timeout=timeout) as r:
            data = r.read()

        root = ET.fromstring(data)
        channel = root.find(".//channel") or root  # fallback if no <channel>

        parts = []
        for it in channel.findall(".//item"):
            title = (it.findtext("title") or "").strip()
            author = (it.findtext("author") or "").strip()
            pub_date = (it.findtext("pubDate") or "").strip()
            description = (it.findtext("description") or "").strip()

            entry_str = (
                f"Title: {title}\n"
                f"Author: {author}\n"
                f"Date: {pub_date}\n"
                f"Description: {description}"
            )

            parts.append(entry_str)
            self.items.append({
                "title": title,
                "author": author,
                "pubDate": pub_date,
                "description": description
            })

        # join into single blob of text
        self.text = "\n\n".join(parts)

system_prompt = (
    """You are an impartial analyst. Assess the current national security threat to Estonia from Russia (conventional or hybrid) or Russia-influenced unrest. Use only the information provided by the user. Be concise, factual, and evidence-driven.

    Output (JSON only)
    {
      "assessment_time_utc": "<ISO8601>",
      "overall_threat_level": "LOW|MODERATE|SUBSTANTIAL|SEVERE|CRITICAL",
      "confidence": 0.0,
      "dimension_scores": {
        "military_posture": 0.0,
        "cross_border_coercion": 0.0,
        "cyber": 0.0,
        "disinformation": 0.0,
        "domestic_unrest": 0.0,
        "critical_infra": 0.0
      },
      "key_signals": ["", ""],
      "notes": ""
    }
    
    Threat Level Definitions
    
    LOW – Hostile action highly unlikely. Only background noise.
    
    MODERATE – Possible but not likely. Mild cyber/disinfo or routine border probes.
    
    SUBSTANTIAL – Likely. Sustained cyber, disinfo, or coercive actions; some GPS jamming.
    
    SEVERE – Highly likely. Coordinated multi-vector actions or sabotage indicators.
    
    CRITICAL – Imminent or ongoing attack. Rapid force movement, destructive cyber, verified plots.
    
    Scoring (0–5)
    
    0 = no threat signal, 5 = extreme signal.
    Overall level ≈ weighted average:
    LOW (<1), MODERATE (1–1.7), SUBSTANTIAL (1.8–2.6), SEVERE (2.6–3.3), CRITICAL (≥3.4).
    If any dimension ≥4.5 and corroborated → escalate to CRITICAL."""
)

def make_user_prompt(feed: "RssFeed") -> str:
    """Format user prompt from RssFeed object."""
    return (
        f"""You are receiving the latest intelligence and situational indicators for Estonia.
        Score the threat level based on the provided facts. Facts:
        {feed.text}
        Please return a JSON assessment."""
        )


# Step 2: Make the messages list

def make_messages(feed: "RssFeed"):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": make_user_prompt(feed)},
    ]
# Step 3: Call Ollama on my machine

import ollama

#response = ollama.chat(model=MODEL, messages=messages)
#print(response['message']['content'])

def assess_security_threat(url: str) -> str:
    """Fetch feed, send to GPT, return score + assessment."""
    feed = RssFeed(url)
    response = ollama.chat(
        #model="deepseek-r1:1.5b",
        model="llama3.2:latest",
        messages=make_messages(feed),
    )
    #return response.choices[0].message.content
    return response['message']['content']

# Step 4: print the result
result = assess_security_threat("https://news.postimees.ee/rss")
print(result)

```json
{
  "assessment_time_utc": "2025-09-20T00:00:00+02:00",
  "overall_threat_level": "SUBSTANTIAL",
  "confidence": 0.8,
  "dimension_scores": {
    "military_posture": 3.9,
    "cross_border_coercion": 2.4,
    "cyber": 2.1,
    "disinformation": 3.5,
    "domestic_unrest": 2.0,
    "critical_infra": 1.9
  },
  "key_signals": ["Russian fighter jets entering Estonian airspace without authorization", "Kremlin's propaganda flow towards Estonia increased in August"],
  "notes": "Increasing tensions between Russia and NATO, including the recent activity aimed to bolster eastern flank, suggest a sustained threat level. The Russian Orthodox Church's support for the Kremlin's war against Ukraine adds to the concern."
}
```

This assessment scores the overall threat level as SUBSTANTIAL due to the cumulative presence of several indicators:

1.  Three Russian fighter jets entering Estonian airspace without authorization, indicating a potential intent to test or challenge Estonian air defen